Sales Analysis

In [1]:
import pandas as pd
#df=pd.read_csv('sales_data.csv')

Loading data


In [ ]:
FILE_NAME = "test_data/sales_data.csv"

try:

    df=pd.read_csv(FILE_NAME)
    print("Data Loaded Successfully")

except FileNotFoundError:
    print(f"Error:File'{FILE_NAME}'was not found")
    raise SystemExit

except pd.errors.EmptyDataError:
    print("Error: The CSV file is empty.")
    raise SystemExit

except Exception as error:
    print(f"Error while loading the dataset:{error}")
    raise SystemExit


Data Loaded Successfully


In [72]:
if df.empty:
    print("Error: The dataset contains no records.")
    raise SystemExit

#Validating Columns

In [74]:
required_columns=[
    "Date",
    "Product",
    "Quantity",
    "Price",
    "Customer_ID",
    "Region",
    "Total_Sales"
]

missing_columns=[
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    print("Error:Required columns are missing:")

    for column in missing_columns:
        print(f"{column}")

        raise SystemExit

Unexpected Columns


In [75]:
unexpected_columns=[
    column for column in df.columns
    if column not in required_columns
]

if unexpected_columns:
    print("Warning: Unexpected columns found:")

    for columns in unexpected_columns:
        print(f"-{columns}")
        

Missing values

In [76]:
missing_values =df.isnull().sum()

print("\n======MISSING VALUES=====")
print(missing_values)

if missing_values.sum() == 0:
    print("No missing values found")
else:
    print("Warning: Missing values detected")


======MISSING VALUES=====
Date           0
Product        0
Quantity       0
Price          0
Customer_ID    0
Region         0
Total_Sales    0
dtype: int64
No missing values found


Checking Duplicate Details

In [77]:
duplicate_count=df.duplicated().sum()

print("\n =====DUPLICATE RECORDS====")
print(f"Duplicate rows:{duplicate_count}")

if duplicate_count>0:
    print("Warning: Duplicate records detected")

else:
    print("No duplicate records found.")


 =====DUPLICATE RECORDS====
Duplicate rows:0
No duplicate records found.


Validate numeric columns

In [78]:
numeric_columns=[
    "Quantity",
    "Price",
    "Total_Sales"
]

for column in numeric_columns:
    df[column]=pd.to_numeric(df[column],errors="coerce")



invalid_numeric_values=df[numeric_columns].isnull().sum()

print("\n===== INVALID NUMERIC VALUES ======")
print(invalid_numeric_values)


===== INVALID NUMERIC VALUES ======
Quantity       0
Price          0
Total_Sales    0
dtype: int64


VALIDATE QUANTITY

In [79]:
invalid_quantity=df["Quantity"] <= 0

if invalid_quantity.any():
    print("Warning: Invalid quantity values found")
    print(df.loc[invalid_quantity,["Product","Quantity"]])
else:
    print("Quantity validation passed")

  Product  Quantity
0   Phone         0


VALIDATE PRICE

In [57]:
invalid_price=df["Price"] <=0

if invalid_price.any():
    print("Warning: Invalid price values found")
    print(df.loc[invalid_price,["Product","Price"]])

else:
    print("Price validation passed")

Price validation passed


Validate Product And Region

In [58]:
blank_products=(
    df["Product"].isna() | 
    (df["Product"].astype(str).str.strip() == "")

)

if blank_products.any():
    print("Warning: Black product names found")
else:
    print("Product validation passed")



Product validation passed


Blank Regions

In [59]:
blank_regions =(
    df["Region"].isna() |
    (df["Region"].astype(str).str.strip() == "")

)

if blank_regions.any():
    print("Warning:Blank regions found")
else:
    print("Region validation passed")

Region validation passed


VALIDATING the DATE

In [63]:
df["Date"]=pd.to_datetime(
    df["Date"],
    errors="coerce"
)

In [64]:
invalid_dates= df["Date"].isna().sum()

if invalid_dates > 0:
    print(f"Warning:{invalid_dates} invalid date(s) found")

else:
    print("Date validation passed")

IMPORTANT VALDATIONS

In [65]:
df["Quantity"]*df["Price"]

0     261100
1      61624
2      43492
3      30895
4     318680
       ...  
95    166160
96      7647
97    135980
98     30717
99    116880
Length: 100, dtype: int64

In [66]:
df["Total_Sales"]

0     999999
1      61624
2      43492
3      30895
4     318680
       ...  
95    166160
96      7647
97    135980
98     30717
99    116880
Name: Total_Sales, Length: 100, dtype: int64

In [67]:
df["Calculated_Sales"] =(
    df["Quantity"]*df["Price"]
)

df["Sales_Match"] = (
    df ["Calculated_Sales"] == df["Total_Sales"]
)

In [68]:
mismatched_sales=(~df["Sales_Match"]).sum()

print("\n =====SALES VALIDATION =====")
print(f"Sales mismatched: {mismatched_sales}")

if mismatched_sales >0:
    print("Warning: Total_Sales does not match  Quantity * price")
    print(
        df.loc[
            ~df["Sales_Match"],
            [
                "Product",
                "Quantity",
                "Price",
                "Total_Sales",
                "Calculated_Sales"

            ]
        ]
    )
else:
    print("Sales calculation validation pssesd")


 =====SALES VALIDATION =====
Sales mismatched: 1
  Product  Quantity  Price  Total_Sales  Calculated_Sales
0   Phone         7  37300       999999            261100


In [26]:
df=df.drop_duplicates()

For products

In [27]:
df["Products"]=(
    df["Product"]
    .astype(str)
    .str.strip()
    .str.title()
)

For Region

In [28]:
df["Region"]=(
    df["Region"]
    .astype(str)
    .str.strip()
    .str.title()
)

ACTUAL ANALYSIS  
Total Revenue

In [29]:
total_revenue =df["Total_Sales"].sum()
print(total_revenue)

12365048


In [30]:
total_quantity=df["Quantity"].sum()
print(total_quantity)

478


Average Sale

In [31]:
average_sale=df["Total_Sales"].mean()
print(average_sale)

123650.48


Highest Sale

In [32]:
highesst_sale= df["Total_Sales"].max()
print(highesst_sale)

373932


Lowest Sale

In [33]:
lowest_sale=df["Total_Sales"].min()
print(lowest_sale)

6540


Product Analysis

In [34]:
product_sales=(
    df.groupby("Product")["Total_Sales"]
    .sum()
    .sort_values(ascending=False)
  
)
print("\n ==== SALES BY PRODUCT =====")
print(product_sales)


 ==== SALES BY PRODUCT =====
Product
Laptop        3889210
Tablet        2884340
Phone         2859394
Headphones    1384033
Monitor       1348071
Name: Total_Sales, dtype: int64


In [35]:
if not product_sales.empty:
    best_product=product_sales.index[0]
    best_product_sales=product_sales.iloc[0]
else:
    best_product="N/A"
    best_product_sales=0

Region analysis


In [36]:
region_sales=(
    df.groupby("Region")["Total_Sales"]
    .sum()
    .sort_values(ascending=False)
)

print("\n ===== SALES B REGION ======")
print(region_sales)


 ===== SALES B REGION ======
Region
North    3983635
South    3737852
East     2519639
West     2123922
Name: Total_Sales, dtype: int64


In [80]:
total_revenue = df["Total_Sales"].sum()

In [81]:
print(f"Total Revenue: ₹{total_revenue:,.2f}")

Total Revenue: ₹12,365,048.00
